# Backtest: Weighted Long-Only Portfolio

Simple weighted buy-and-hold backtest. You provide tickers and weights; the notebook
computes historical performance metrics using **log returns** throughout.

**Workflow:**
1. Run `notebooks/01_project_walkthrough/explore_selection_methods.ipynb` to pick your ETFs.
2. Run `notebooks/01_project_walkthrough/explore_allocation_methods.ipynb` to compute weights.
3. Edit the basket cell below and run this notebook.

**Metrics reported (all log-return based):**
- `ann_return`      = mean(log_ret_daily) * 252
- `ann_vol`         = std(log_ret_daily) * sqrt(252)
- `sharpe`          = (ann_return - risk_free) / ann_vol
- `total_log_return`= sum of all daily portfolio log returns
- `max_drawdown`    = peak-to-trough loss on the log-return wealth path
- `calmar`          = ann_return / max_drawdown

In [1]:
import pathlib
import sys

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / "GUIDE_ROOT.md").exists():
        for _candidate in [_p / "src", _p]:
            if str(_candidate) not in sys.path:
                sys.path.insert(0, str(_candidate))
        PROJECT_ROOT = _p
        break

from backtesting.buy_and_hold import run_weighted_backtest

START_DATE = "2018-01-01"
END_DATE = "2025-12-31"
INITIAL_CAPITAL = 30_000
RISK_FREE = 0.05

print("Ready. Edit TICKERS and WEIGHTS below, then run the results cell.")

Ready. Edit TICKERS and WEIGHTS below, then run the results cell.


## Define Your Basket

Edit `TICKERS` and `WEIGHTS` to match whatever allocation you want to evaluate.
Weights must sum to 1.0 and all values must be >= 0 (long-only).

The default below uses the 10-ETF basket with equal weight (10% each).

In [2]:
# Edit this cell to test different allocations.
# Weights must sum to 1.0.  All values must be >= 0.

TICKERS = [
    "VOO",  # US large-cap equity
    "VEA",  # Developed ex-US equity
    "VWO",  # Emerging market equity
    "TLT",  # US long-term Treasury bonds
    "LQD",  # Investment-grade corporate bonds
    "IAU",  # Gold
    "DBA",  # Agriculture commodities
    "VNQ",  # US REITs
    "XLP",  # Consumer staples
    "VWOB",  # EM sovereign bonds
]

WEIGHTS = [
    0.10,  # VOO
    0.10,  # VEA
    0.10,  # VWO
    0.10,  # TLT
    0.10,  # LQD
    0.10,  # IAU
    0.10,  # DBA
    0.10,  # VNQ
    0.10,  # XLP
    0.10,  # VWOB
]

print("Basket:")
for t, w in zip(TICKERS, WEIGHTS):
    print(f"  {t:<6}  {w:.2%}")
print(f"Weight sum: {sum(WEIGHTS):.4f}")

Basket:
  VOO     10.00%
  VEA     10.00%
  VWO     10.00%
  TLT     10.00%
  LQD     10.00%
  IAU     10.00%
  DBA     10.00%
  VNQ     10.00%
  XLP     10.00%
  VWOB    10.00%
Weight sum: 1.0000


## Backtest Results

Runs the weighted log-return backtest and displays summary metrics plus equity curve.

In [3]:
metrics, equity = run_weighted_backtest(
    tickers=TICKERS,
    weights=WEIGHTS,
    start_date=START_DATE,
    end_date=END_DATE,
    initial_capital=INITIAL_CAPITAL,
    risk_free=RISK_FREE,
)

print("\n" + "=" * 55)
print("BACKTEST RESULTS")
print("=" * 55)
print(f"  Period        : {metrics['start_date']}  to  {metrics['end_date']}")
print(f"  Trading days  : {metrics['n_trading_days']}")
print(f"  Initial cap   : ${metrics['initial_capital']:,.0f}")
print(f"  Final value   : ${metrics['final_value']:,.2f}")
print()
print(f"  Total log ret : {metrics['total_log_return']:.4f}")
print(f"  Ann. return   : {metrics['ann_return']:.4f}  ({metrics['ann_return']:.2%})")
print(f"  Ann. vol      : {metrics['ann_vol']:.4f}  ({metrics['ann_vol']:.2%})")
print(f"  Sharpe        : {metrics['sharpe']:.4f}")
print(
    f"  Max drawdown  : {metrics['max_drawdown']:.4f}  ({metrics['max_drawdown']:.2%})"
)
print(f"  Calmar        : {metrics['calmar']:.4f}")
print("=" * 55)

# Equity curve + drawdown chart
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, constrained_layout=True)

axes[0].plot(equity["date"], equity["portfolio_value"], color="#003f5c", linewidth=1.8)
axes[0].set_title(f"Portfolio Value  (initial ${INITIAL_CAPITAL:,})", fontsize=12)
axes[0].set_ylabel("USD")
axes[0].grid(alpha=0.3)
axes[0].axhline(
    INITIAL_CAPITAL, color="grey", ls="--", alpha=0.5, label="Initial capital"
)
axes[0].legend(fontsize=9)

axes[1].fill_between(
    equity["date"], equity["drawdown"] * 100, 0, color="#bc5090", alpha=0.4
)
axes[1].plot(equity["date"], equity["drawdown"] * 100, color="#bc5090", lw=0.8)
axes[1].set_title("Drawdown (%)", fontsize=12)
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Drawdown (%)")
axes[1].grid(alpha=0.3)

import matplotlib.dates as mdates

axes[1].xaxis.set_major_locator(mdates.YearLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.show()


BACKTEST RESULTS
  Period        : 2018-01-02  to  2025-12-31
  Trading days  : 2011
  Initial cap   : $30,000
  Final value   : $52,028.68

  Total log ret : 0.5506
  Ann. return   : 0.0690  (6.90%)
  Ann. vol      : 0.1045  (10.45%)
  Sharpe        : 0.1821
  Max drawdown  : 0.2148  (21.48%)
  Calmar        : 0.3214
